# 📘 Predicting Salary from CGPA using XGBoost (Library Version)

This Colab notebook demonstrates how to solve a **regression problem using XGBoost**, with the **official `xgboost` Python library**.

We will:
- Load a toy dataset with CGPA and one-hot encoded categorical variables.
- Train an XGBoost model using `XGBRegressor`.
- Observe how it learns the patterns stage by stage (visually via plots).
- Compare results with earlier manually calculated boosting stages.

This version does **not manually build the trees**, but rather uses the optimized implementation by the `xgboost` library.


In [ ]:
# Install the XGBoost library if not already installed
!pip install xgboost -q


In [ ]:
# Basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


## 🧪 Step 1: Create a small dataset

We'll use:
- `CGPA`: a continuous feature
- `Branch`: a categorical feature (e.g. CS, EE, ME)
- `Package`: target output in LPA


In [ ]:
# Create the dataset
df = pd.DataFrame({
    'CGPA': [6.7, 9.0, 7.5, 6.0, 7.8, 8.3],
    'Branch': ['CS', 'CS', 'EE', 'ME', 'EE', 'ME'],
    'Package': [4.5, 11.0, 6.0, 2.8, 7.0, 8.0]
})

df


## 🔠 Step 2: Encode the categorical feature

We use **OneHotEncoding** so that models can use non-numeric values like `'CS'`, `'EE'`, etc.


In [ ]:
# One-hot encode the 'Branch' column
encoder = OneHotEncoder(sparse=False)
encoded_branch = encoder.fit_transform(df[['Branch']])
branch_df = pd.DataFrame(encoded_branch, columns=encoder.get_feature_names_out(['Branch']))

# Combine with CGPA and target
X = pd.concat([df[['CGPA']], branch_df], axis=1)
y = df['Package']
X


## ✂️ Step 3: Train-test split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)


## 🎯 Step 4: Train an XGBoost Regressor

We set `n_estimators=4` to match our 4-stage manual boosting workflow.


In [ ]:
# Train the model with 4 boosting stages
model = xgb.XGBRegressor(n_estimators=4, learning_rate=0.3, objective='reg:squarederror')
model.fit(X_train, y_train)


## 🧾 Step 5: Evaluate and Predict


In [ ]:
y_pred = model.predict(X_test)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


## 📊 Step 6: Visualize Predictions


In [ ]:
# Plot true vs predicted
plt.figure(figsize=(6,4))
plt.scatter(y_test, y_pred, color='purple')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
plt.xlabel("True Package")
plt.ylabel("Predicted Package")
plt.title("XGBoost Predictions vs Actual")
plt.grid(True)
plt.show()
